# 46 · Bounded raw-event adapter audit

Replay recorded heart-rate and motion events through a bounded adapter. Use checksum-verified raw sensor files for development participants.
The replay driver sorts each recorded stream, merges the streams, and counts events outside the common session interval.
The adapter accepts events in timestamp order within each stream. It collapses exact duplicates and raises on conflicting duplicates.

Each stream has a caller-supplied watermark that can only advance. A watermark of `w` promises that future events have timestamps at or above `w`.
Events at `w` are allowed. With zero allowed lateness, the adapter rejects any event below its stream's watermark.
The consumer can pop one complete epoch `[t, t + 30)` once both watermarks reach its end.
The configured limit is 8,192 buffered events across both streams. Overflow raises an error without silently losing buffered observations.
With an explicit session end, drain only complete epochs and count any events discarded from the partial tail.

Compare both replay chunk sizes with batch features at absolute tolerance 1e-8. Check epoch eligibility and sensor segment identity.
From the first chunk size, run frozen inference once across all five MPS folds. Compare probabilities with the saved sensor-only predictions
from [notebook 45](45_sensor_only_frozen_replay.ipynb) at absolute tolerance 2e-6. The audit uses development sensor observations and fixed models.
Training and reserved final-test access are outside the audit's scope.

## Locate the checkout and set run options

**What:** Resolve the project root and set the setup and audit flags.

**Why:** With both flags disabled, the notebook reads the plan and displays the status of existing results.
Set `SLEEP_CLASSIFIER_ROOT` to specify a different checkout.

In [ ]:
import logging
import os
import shutil
import sys
from pathlib import Path

from IPython import get_ipython
from IPython.display import display

logging.basicConfig(level=logging.INFO, stream=sys.stdout, force=True)
logger = logging.getLogger("raw_event_audit_notebook")
ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "pyproject.toml").is_file() and (path / "src/sleep_classifier").is_dir()),
    Path.cwd() / "sleep-classifier",
)
ROOT = Path(os.environ.get("SLEEP_CLASSIFIER_ROOT", str(ROOT))).expanduser().resolve()

REPOSITORY_URL = "https://github.com/rbl-bidilatimotei/sleep-classifier.git"
BRANCH = "feature/bounded-raw-event-adapter"
RUN_SETUP = False
RUN_AUDIT = False

logger.info("Repository=%s setup=%s audit=%s", ROOT, RUN_SETUP, RUN_AUDIT)

## Set up the environment when requested

**What:** Clone the feature branch if necessary. Install uv if needed, then sync the locked GRU, notebook, and development groups.

**Why:** Inexact sync preserves other installed groups. Set `RUN_SETUP = True` and run the flag and setup cells.
Then select the checkout's `.venv` kernel if necessary. Restore `RUN_SETUP = False` before saving.

For a fresh audit, provide the verified raw sensor files and the saved reference artifacts specified by the audit configuration.
Keep their configured paths relative to the checkout.

In [ ]:
if RUN_SETUP:
    if not ROOT.exists():
        !git clone --branch {BRANCH} {REPOSITORY_URL} "{ROOT}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("Repository clone failed")
    if shutil.which("uv") is None:
        installer = ROOT / "artifacts/uv-install.sh"
        installer.parent.mkdir(parents=True, exist_ok=True)
        !curl -LsSf https://astral.sh/uv/install.sh -o "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv download failed")
        !sh "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv installation failed")
        os.environ["PATH"] = str(Path.home() / ".local/bin") + os.pathsep + os.environ["PATH"]
    %cd "{ROOT}"
    !uv sync --locked --inexact --group gru --group notebook --group dev
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Environment setup failed")
else:
    logger.info("Environment setup skipped")


## Inspect the audit plan

**What:** Show the configured development scope, chunk sizes, buffer limit, and inference device.

**Why:** The plan records the comparison settings before reading sensor observations or loading model weights.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from sleep_classifier.raw_event_audit import RawEventAudit

audit = RawEventAudit(ROOT / "configs/raw-event-adapter.yaml")
plan = audit.plan()
logger.info("Audit config=%s output=%s", audit.config_path, audit.output_dir)
display(pd.DataFrame([{
    "development_participants": len(plan["participants"]),
    "chunk_sizes": plan["chunk_sizes"],
    "max_buffered_events": plan["max_buffered_events"],
    "device": plan["device"],
}]))

## Run the audit when requested

**What:** Validate readiness and run the audit when `RUN_AUDIT` is enabled and the output directory is absent.

**Why:** Fresh execution reads verified sensor records and runs frozen inference. The following cell reads existing reports.

Set `RUN_AUDIT = True` and run the flag, plan, and audit cells. Restore `RUN_AUDIT = False` before saving.
From the checkout root, use the same command to run the audit:

```bash
uv run --locked --inexact --group gru --group notebook python -u \
  -m sleep_classifier.raw_event_audit --config configs/raw-event-adapter.yaml --run
```

Use a new config and output path for an intentional repeat.

In [ ]:
if RUN_AUDIT and not audit.output_dir.exists():
    audit.validate_run_start()
    logger.info("Run audit config=%s output=%s", audit.config_path, audit.output_dir)
    %cd "{ROOT}"
    !uv run --locked --inexact --group gru --group notebook python -u \
        -m sleep_classifier.raw_event_audit --config "{audit.config_path}" --run
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Raw-event adapter audit failed")
elif audit.output_dir.exists():
    logger.info("Read the existing audit report at %s", audit.output_dir)
else:
    logger.info("Audit skipped because RUN_AUDIT is disabled")

## Load the audit status

**What:** Read the report status and display the training and reserved test access flags.

**Why:** The comparison tables require a complete report. Pending or partial results remain visible as status messages.

In [ ]:
report = audit.read_report()
report_ready = report["status"] == "passed"
logger.info("Audit status=%s output=%s", report["status"], audit.output_dir)
if report["status"] == "pending":
    logger.info("Audit pending. Enable RUN_AUDIT when its inputs are available")
elif report["status"] == "partial":
    logger.info("The output directory exists but its report is incomplete")
if report_ready:
    display(pd.DataFrame([{
        "training_performed": report["training_performed"],
        "test_observations_read": report["test_observations_read"],
    }]))

## Compare participant replay with batch preparation

**What:** Show epoch counts, feature and segment parity, and event accounting by chunk size.

**Why:** Equal epoch counts can hide differences in numerical features or eligible histories.
Historical feature parity uses the saved [notebook 44](44_label_free_sensor_audit.ipynb) sensor preparation.
Event conservation checks that every submitted event was used, collapsed as an exact duplicate, or counted in the discarded tail.

In [ ]:
if report_ready:
    participants = pd.DataFrame(report["participants"]).set_index(["participant_id", "chunk_size"])
    measures = [
        "epochs", "eligible_epochs", "feature_maximum_difference", "feature_mismatches",
        "historical_feature_maximum_difference", "historical_feature_mismatches",
        "eligibility_matches", "segments_match", "event_conservation_matches",
        "peak_buffered_events", "duplicate_events", "discarded_tail_events",
        "filtered_pre_session_events", "filtered_after_session_events", "elapsed_seconds",
    ]
    display(participants.loc[:, [key for key in measures if key in participants]].sort_index())

## Compare observed buffer peaks and replay times

**What:** Plot peak buffered events and elapsed replay time for each participant at both chunk sizes.

**Why:** These observations show how chunk size affected the recorded replay.
The buffer limit applies to retained adapter events. Elapsed times describe this audit run.
A deployment latency or process-memory benchmark requires separate measurement.

In [ ]:
if report_ready:
    fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
    participants["peak_buffered_events"].unstack("chunk_size").plot.bar(ax=axes[0])
    axes[0].axhline(
        plan["max_buffered_events"], color="black", linestyle="--", label="Configured limit",
    )
    axes[0].set(ylabel="Buffered events", title="Peak adapter buffer")
    axes[0].legend(title="Replay chunk size")

    participants["elapsed_seconds"].unstack("chunk_size").plot.bar(ax=axes[1])
    axes[1].set(xlabel="Development participant", ylabel="Seconds", title="Observed replay time")
    axes[1].legend(title="Replay chunk size")
    fig.tight_layout()
    plt.show()

## Compare frozen predictions

**What:** Show prediction differences, history and reset agreement, checkpoint hashes, and model integrity by fold.

**Why:** These checks test whether feeding the same frozen models with streamed features preserves the saved development predictions.
A passing parity check establishes equivalence within the configured tolerances for this replay.

In [ ]:
if report_ready:
    display(pd.DataFrame(report["prediction_parity"]).set_index("fold")[[
        "epochs", "maximum_probability_difference", "class_disagreements",
        "history_lengths_match", "reset_reasons_match", "weights_unchanged",
        "saved_transforms_unchanged", "gradients_created", "evaluation_mode",
        "checkpoint_sha256",
    ]].sort_index())

## Completed cohort audit · 2026-11-26

Both chunk sizes passed across all twenty-six development participants.
All 24,738 completed epochs and 23,674 eligible epochs matched fresh batch preparation and the saved sensor projection exactly.
Each chunking processed 36,133,288 accepted input events, including 31,609 exact duplicates and 18,852 discarded tail events.
Event conservation passed in every session. Pending queues peaked at 2,131 and 3,026 events, below the 8,192 limit.

All five frozen models reproduced all 23,674 notebook-45 probabilities exactly, with zero class disagreements.
History lengths and reset reasons matched. Model weights and fitted transforms stayed unchanged, with evaluation mode and no gradients.
No training or final-test observation reads occurred. This verifies ordered recorded-event replay, not live device timing or total process memory.
Keep RUN_AUDIT false to inspect the immutable full report.
See [the measured findings](../docs/experiments/raw-event-adapter-2026-11-26/README.md).


## Export a report-only HTML copy

**What:** Execute report cells in a fresh kernel and save the HTML output.

**Why:** Export skips setup, audit execution, and recursive export. The source notebook stays output-free.

In [ ]:
%cd "{ROOT}"
!uv run --locked --inexact --group gru --group notebook python scripts/execute_notebook.py \
    --notebook notebooks/46_raw_event_adapter.ipynb
if get_ipython().user_ns["_exit_code"] != 0:
    raise RuntimeError("Raw-event adapter notebook export failed")
